In [1]:
from datetime import datetime

print(f"Timestamp: {datetime.now()}")

Timestamp: 2025-11-17 14:34:06.145868


Okay so basically we have four branches of the data. Nuclear and whole cell, and with and without the transgenes. It makes sense to make sure the scVI latent rep etc. is not being influenced by the GFP etc.

# Train scVI model

In [2]:
import os
from pathlib import Path
import scanpy as sc
from scipy.sparse import issparse, csr_matrix
import scvi
import numpy as np
from collections import Counter
import anndata as ad
import matplotlib.pyplot as plt
from lightning.pytorch import seed_everything
import random
import torch
import sys
import session_info

In [3]:
random.seed(0)
seed_everything(0)

scvi.settings.seed = 0
scvi.settings.num_workers = 32

Global seed set to 0
Global seed set to 0


## Set paths, import data

In [4]:
base_dir = Path('/home/workspace/projects/drg')

input_dir = base_dir / 'data/h5ad/export_01/04_filtered'
scvi_dir = base_dir / 'scvi/01_model'
output_dir = base_dir/ 'data/h5ad/export_02/01_scvi'

os.makedirs(output_dir, exist_ok=True)

In [5]:
path_whole   = input_dir / "adata-filtered_whole.h5ad"
path_nuclear = input_dir / "adata-filtered_nuclear.h5ad"

In [6]:
adata = sc.read_h5ad(path_whole)
bdata = sc.read_h5ad(path_nuclear)

## Prepare adata

In [7]:
# Save counts layer for scVI and downstream processing
adata.X = adata.layers['counts'].copy()
bdata.X = bdata.layers['counts'].copy()

In [8]:
for n, a in zip(["adata", "bdata"], [adata, bdata]):
    if not issparse(a.X):
        a.X = csr_matrix(a.X)
        print(f"Converted {n} to sparse CSR")

Converted bdata to sparse CSR


In [9]:
adata.X[:5, :5].toarray()

array([[ 0.,  2.,  0.,  2.,  0.],
       [ 3., 20.,  1., 25.,  0.],
       [ 1.,  1.,  5., 13.,  0.],
       [ 0.,  1.,  0.,  0.,  0.],
       [ 0.,  4.,  0.,  9.,  0.]], dtype=float32)

In [10]:
bdata.X[:5, :5].toarray()

array([[0., 2., 0., 2., 0.],
       [2., 3., 0., 1., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.]])

In [11]:
print('adata.X is sparse:', issparse(adata.X))
print('adata.X has only whole numbers:', np.all(adata.X.data == np.round(adata.X.data)))  # True if all values are whole numbers

adata.X is sparse: True
adata.X has only whole numbers: True


In [12]:
adata_all = adata.copy()
bdata_all = bdata.copy()

In [13]:
def remove_transgenes(adata, pattern="Seq"):
    """
    Remove the five transgenes, which all have "Seq" string
    """
    print(f"\nProcessing: {getattr(adata, 'uns', {}).get('name', 'AnnData')}")
    print(f"Initial # genes: {adata.n_vars}")
    
    transgenes = adata.var_names[adata.var_names.str.contains(pattern, case=False, na=False)]
    print(f"Found {len(transgenes)} transgenes: {list(transgenes)[:10]}{' ...' if len(transgenes) > 10 else ''}")

    adata_filtered = adata[:, ~adata.var_names.isin(transgenes)].copy()
    print(f"Remaining genes: {adata_filtered.n_vars}")

    return adata_filtered

In [14]:
adata_nt = remove_transgenes(adata, pattern="Seq")
bdata_nt = remove_transgenes(bdata, pattern="Seq")


Processing: AnnData
Initial # genes: 480
Found 5 transgenes: ['Cre_Seq5', 'EGFP_Seq1', 'Flp_Seq4', 'mCherry_Seq3', 'tdTomato_Seq2']
Remaining genes: 475

Processing: AnnData
Initial # genes: 480
Found 5 transgenes: ['Cre_Seq5', 'EGFP_Seq1', 'Flp_Seq4', 'mCherry_Seq3', 'tdTomato_Seq2']
Remaining genes: 475


# scVI

* Model A - adata_all: whole cell, all genes
* Model B - adata_nt: whole cell, no transgenes
* Model C - bdata_all: nuclear, all genes
* Model D - bdata_nt: nuclear, no transgenes

In [ ]:
def train_scvi_model(adata, model_name, scvi_dir, output_dir, latent_key_suffix=""):
    """
    Train sequential scVI models
    """
    print(f"\n Training {model_name}")

    # Setup for scVI
    scvi.model.SCVI.setup_anndata(
        adata,
        layer="counts",
        batch_key="slide"
    )

    # Initialize model
    model = scvi.model.SCVI(
        adata,
        n_layers=2,
        n_latent=30,
        gene_likelihood="nb"
    )

    # Train model
    model.train(
        early_stopping=True,
        #accelerator="gpu",
        early_stopping_patience=5,
        early_stopping_min_delta=1e-4,
        enable_progress_bar=True
    )

    # Save model
    model.save(scvi_dir, prefix=f"{model_name}_")

    # Plot ELBO loss curves
    plt.plot(model.history["elbo_train"], label="Train ELBO Loss")
    plt.plot(model.history["elbo_validation"], label="Validation ELBO Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.title(f"{model_name} SCVI Training Loss")
    plt.tight_layout()
    plt.show()

    # Add latent representation
    latent_key = f"X_scVI{latent_key_suffix}"
    adata.obsm[latent_key] = model.get_latent_representation(adata).astype(np.float32)

    # Save AnnData
    os.makedirs(output_dir, exist_ok=True)
    filename = os.path.join(output_dir, f"{model_name}_adata-scvi.h5ad")
    adata.write_h5ad(filename, compression="gzip")

    print(f"Saved model + latent to {filename} ({latent_key})")

    return model, adata

In [ ]:
train_scvi_model(adata_all, "ModelA_whole_all", scvi_dir, output_dir)
train_scvi_model(adata_nt,  "ModelB_whole_noTG", scvi_dir, output_dir)
train_scvi_model(bdata_all, "ModelC_nuclear_all", scvi_dir, output_dir)
train_scvi_model(bdata_nt,  "ModelD_nuclear_noTG", scvi_dir, output_dir)

## Session info

In [ ]:
print('active IDE: sc-spatial-gpu')
print('active conda environment:', os.path.basename(sys.prefix))
session_info.show()